# Pemodelan dan Simulasi Sistem Dinamik SEIRV untuk Menganalisis Pengaruh Laju Vaksinasi terhadap Kasus Aktif COVID-19 di Provinsi Jawa Barat

**Mata Kuliah:** Pemodelan dan Simulasi (UTS Teori & Praktikum)  
**Dataset:** `DataSetCvd.xlsx` (Sheet `WJ_200304_221002` & `INA_Vacc`)  
**Metode:** Ordinary Differential Equations (ODE) Kompartemen *Susceptible-Exposed-Infected-Recovered-Vaccinated* (SEIRV) & Optimasi Parameter *Curve Fitting* (L-BFGS-B)

## 1. Import Library dan Persiapan Dataset
Pastikan file `DataSetCvd.xlsx` sudah diunggah ke sesi Google Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error, r2_score

# Konfigurasi visualisasi grafik
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['font.size'] = 11

# Membaca dataset Jawa Barat dan Vaksinasi Nasional
file_path = 'DataSetCvd.xlsx'
df_wj = pd.read_excel(file_path, sheet_name='WJ_200304_221002')
df_vacc = pd.read_excel(file_path, sheet_name='INA_Vacc')

df_wj['Tanggal'] = pd.to_datetime(df_wj['Tanggal'])
df_vacc['Date'] = pd.to_datetime(df_vacc['Date'])

# Penggabungan dataset berdasarkan tanggal
df = pd.merge(df_wj, df_vacc[['Date', 'Vacc_1', 'Vacc_2']], left_on='Tanggal', right_on='Date', how='left')
print(f"Total data harian tersedia: {len(df)} baris ({df['Tanggal'].min().date()} s/d {df['Tanggal'].max().date()})")
df[['Tanggal', 'Confirmed', 'Confirmed_active', 'Confirmed_recovered', 'Confirmed_deaths', 'Vacc_2']].dropna().head()

## 2. Pra-pemrosesan Data & Penentuan Periode Simulasi
Periode observasi difokuskan pada gelombang akselerasi kasus dan percepatan program vaksinasi di Jawa Barat (**5 Juni 2021 – 10 September 2021**, durasi 98 hari). Proporsi populasi Jawa Barat diestimasi sebesar $17,8\%$ dari total penduduk nasional ($N = 48.274.162$ jiwa berdasarkan Sensus Penduduk BPS).

In [ ]:
# Filter rentang waktu simulasi
start_date = '2021-06-05'
end_date = '2021-09-10'
sub = df[(df['Tanggal'] >= start_date) & (df['Tanggal'] <= end_date)].copy().reset_index(drop=True)

# Interpolasi nilai kosong (missing values)
for col in ['Confirmed_active', 'Confirmed_recovered', 'Confirmed_deaths', 'Vacc_2']:
    sub[col] = sub[col].interpolate().bfill().ffill()

# Estimasi cakupan vaksinasi dosis lengkap di Jawa Barat (~17.8% proporsi nasional)
sub['Vacc_Jabar'] = sub['Vacc_2'] * 0.178

N_total = 48274162.0  # Total populasi Jawa Barat
I_obs = sub['Confirmed_active'].values
R_obs = sub['Confirmed_recovered'].values
V_obs = sub['Vacc_Jabar'].values
t = np.arange(len(sub))

print(f"Jumlah hari simulasi : {len(t)} hari")
print(f"Kasus aktif awal     : {I_obs[0]:,.0f} jiwa")
print(f"Puncak kasus aktif   : {I_obs.max():,.0f} jiwa pada {sub.loc[sub['Confirmed_active'].idxmax(), 'Tanggal'].date()}")

## 3. Formulasi Model Sistem Dinamik SEIRV
Sistem persamaan diferensial biasa (ODE) kompartemen SEIRV dirumuskan sebagai berikut:

$$\frac{dS}{dt} = -\frac{\beta S I}{N_{\text{eff}}} - \nu S$$
$$\frac{dE}{dt} = \frac{\beta S I}{N_{\text{eff}}} - \sigma E$$
$$\frac{dI}{dt} = \sigma E - \gamma I - \delta I$$
$$\frac{dR}{dt} = \gamma I$$
$$\frac{dV}{dt} = \nu S$$

Dengan Bilangan Reproduksi Dasar ($R_0$) dan Bilangan Reproduksi Efektif dengan vaksinasi ($R_v$):
$$R_0 = \frac{\beta}{\gamma + \delta}, \qquad R_v = R_0 \left(\frac{S_0}{N_{\text{eff}}}\right)$$

In [ ]:
def model_seirv(y, t, N_eff, beta, sigma, gamma, delta, nu):
    S, E, I, R, V = y
    dSdt = -(beta * S * I) / N_eff - nu * S
    dEdt = (beta * S * I) / N_eff - sigma * E
    dIdt = sigma * E - gamma * I - delta * I
    dRdt = gamma * I
    dVdt = nu * S
    return [dSdt, dEdt, dIdt, dRdt, dVdt]

# Parameter epidemiologis tetap berdasarkan literatur klinis COVID-19
sigma_fixed = 1.0 / 5.2   # Masa inkubasi rata-rata 5.2 hari
delta_fixed = 0.0012      # Laju mortalitas harian

I0 = I_obs[0]
R0_init = R_obs[0]
V0_init = V_obs[0]

# Fungsi objektif (Mean Squared Error) untuk kalibrasi parameter
def objective_function(params):
    beta, gamma, nu, n_scale = params
    N_eff = N_total * n_scale
    E0 = I0 * 0.85
    R0_s = R0_init * n_scale
    V0_s = V0_init * n_scale
    S0 = max(N_eff - E0 - I0 - R0_s - V0_s, I0 * 5)
    sol = odeint(model_seirv, [S0, E0, I0, R0_s, V0_s], t, args=(N_eff, beta, sigma_fixed, gamma, delta_fixed, nu))
    I_pred = sol[:, 2]
    return np.mean((I_pred - I_obs) ** 2)

# Optimasi parameter dengan metode L-BFGS-B
initial_guess = [0.40, 0.12, 0.004, 0.04]
bounds = [(0.05, 1.5), (0.04, 0.30), (0.0005, 0.04), (0.01, 0.5)]
result = minimize(objective_function, initial_guess, bounds=bounds, method='L-BFGS-B')

beta_opt, gamma_opt, nu_opt, scale_opt = result.x
N_eff_opt = N_total * scale_opt
E0_opt = I0 * 0.85
R0_opt = R0_init * scale_opt
V0_opt = V0_init * scale_opt
S0_opt = max(N_eff_opt - E0_opt - I0 - R0_opt - V0_opt, I0 * 5)

# Simulasi Baseline terkalibrasi
sol_baseline = odeint(model_seirv, [S0_opt, E0_opt, I0, R0_opt, V0_opt], t,
                      args=(N_eff_opt, beta_opt, sigma_fixed, gamma_opt, delta_fixed, nu_opt))
I_pred_base = sol_baseline[:, 2]

# Evaluasi Akurasi Model
rmse = np.sqrt(mean_squared_error(I_obs, I_pred_base))
mape = np.mean(np.abs((I_obs - I_pred_base) / I_obs)) * 100
r2 = r2_score(I_obs, I_pred_base)
R0_number = beta_opt / (gamma_opt + delta_fixed)

print("=== HASIL KALIBRASI PARAMETER MODEL SEIRV ===")
print(f"Laju Transmisi (beta)         : {beta_opt:.4f} /hari")
print(f"Laju Inkubasi (sigma)         : {sigma_fixed:.4f} /hari")
print(f"Laju Kesembuhan (gamma)       : {gamma_opt:.4f} /hari")
print(f"Laju Mortalitas (delta)       : {delta_fixed:.4f} /hari")
print(f"Laju Vaksinasi Baseline (nu)  : {nu_opt:.5f} /hari")
print(f"Bilangan Reproduksi Dasar (R0): {R0_number:.4f}")
print(f"Koefisien Determinasi (R^2)   : {r2:.4f}")
print(f"RMSE                          : {rmse:,.2f} jiwa")
print(f"MAPE                          : {mape:.2f}%")

## 4. Uji Validasi Kurva Aktual vs Simulasi SEIRV

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(sub['Tanggal'], I_obs, 'o-', color='#1e293b', markersize=4, linewidth=1.5, label='Data Aktual Jawa Barat (Confirmed Active)')
plt.plot(sub['Tanggal'], I_pred_base, '-', color='#0284c7', linewidth=3, label=f'Simulasi Model SEIRV (R² = {r2:.3f}, MAPE = {mape:.2f}%)')
plt.title('Validasi Model Sistem Dinamik SEIRV terhadap Data Kasus Aktif COVID-19 Jawa Barat', fontweight='bold', pad=15)
plt.xlabel('Tanggal Observasi')
plt.ylabel('Jumlah Kasus Aktif (Jiwa)')
plt.legend(frameon=True, facecolor='white')
plt.tight_layout()
plt.savefig('validasi_seirv_jabar.png', dpi=300)
plt.show()

## 5. Simulasi Skenario Analisis Kebijakan Laju Vaksinasi ($\nu$)
Menguji sensitivitas penurunan puncak kasus aktif terhadap empat skenario intervensi laju vaksinasi:
1. **Tanpa Vaksinasi ($\nu = 0$)**
2. **Baseline Historis ($\nu = \nu_{\text{opt}}$)**
3. **Percepatan Moderat ($\nu = 3 \times \nu_{\text{opt}}$)**
4. **Percepatan Agresif ($\nu = 6 \times \nu_{\text{opt}}$)**

In [ ]:
scenarios = {
    'Tanpa Vaksinasi (nu = 0)': (0.0, '#dc2626', '--'),
    'Baseline Aktual (1.0x nu)': (nu_opt, '#0284c7', '-'),
    'Percepatan Vaksinasi Moderat (3.0x nu)': (nu_opt * 3.0, '#d97706', '-.'),
    'Percepatan Vaksinasi Agresif (6.0x nu)': (nu_opt * 6.0, '#059669', '-')
}

summary_rows = []
plt.figure(figsize=(13, 6))

for label, (nu_val, color, style) in scenarios.items():
    sol_s = odeint(model_seirv, [S0_opt, E0_opt, I0, R0_opt, V0_opt], t,
                   args=(N_eff_opt, beta_opt, sigma_fixed, gamma_opt, delta_fixed, nu_val))
    I_s = sol_s[:, 2]
    V_s = sol_s[:, 4]
    peak_val = I_s.max()
    peak_day = sub.loc[np.argmax(I_s), 'Tanggal'].date()
    end_val = I_s[-1]
    
    summary_rows.append({
        'Skenario': label,
        'Laju Vaksinasi (nu/hari)': round(nu_val, 5),
        'Puncak Kasus Aktif (Jiwa)': int(round(peak_val)),
        'Tanggal Puncak': str(peak_day),
        'Kasus Aktif Akhir Periode (Jiwa)': int(round(end_val))
    })
    plt.plot(sub['Tanggal'], I_s, linestyle=style, color=color, linewidth=2.5, label=f'{label} | Puncak: {peak_val:,.0f}')

plt.scatter(sub['Tanggal'], I_obs, color='#334155', s=18, alpha=0.6, label='Data Aktual Jawa Barat')
plt.title('Simulasi Pengaruh Laju Vaksinasi terhadap Dinamika Kasus Aktif COVID-19 Jawa Barat', fontweight='bold', pad=15)
plt.xlabel('Tanggal Simulasi')
plt.ylabel('Populasi Terinfeksi Aktif I(t) [Jiwa]')
plt.legend(frameon=True, facecolor='white', loc='upper right')
plt.tight_layout()
plt.savefig('skenario_vaksinasi_seirv.png', dpi=300)
plt.show()

df_summary = pd.DataFrame(summary_rows)
df_summary['Reduksi Puncak vs Tanpa Vaksin (%)'] = (
    (df_summary.loc[0, 'Puncak Kasus Aktif (Jiwa)'] - df_summary['Puncak Kasus Aktif (Jiwa)'])
    / df_summary.loc[0, 'Puncak Kasus Aktif (Jiwa)'] * 100
).round(2)
display(df_summary)

## 6. Dinamika Seluruh Kompartemen (S, E, I, R, V)
Visualisasi perpindahan populasi antar kompartemen pada skenario percepatan vaksinasi.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: S, R, V
axes[0].plot(sub['Tanggal'], sol_baseline[:, 0], label='Susceptible (S)', color='#3b82f6', linewidth=2)
axes[0].plot(sub['Tanggal'], sol_baseline[:, 3], label='Recovered (R)', color='#10b981', linewidth=2)
axes[0].plot(sub['Tanggal'], sol_baseline[:, 4], label='Vaccinated (V)', color='#8b5cf6', linewidth=2)
axes[0].set_title('Dinamika Kompartemen S, R, dan V (Baseline)', fontweight='bold')
axes[0].set_ylabel('Jumlah Populasi Efektif (Jiwa)')
axes[0].legend()

# Subplot 2: E dan I
axes[1].plot(sub['Tanggal'], sol_baseline[:, 1], label='Exposed (E)', color='#f59e0b', linewidth=2)
axes[1].plot(sub['Tanggal'], sol_baseline[:, 2], label='Infected Active (I)', color='#ef4444', linewidth=2)
axes[1].set_title('Dinamika Kompartemen Terpapar (E) dan Terinfeksi Aktif (I)', fontweight='bold')
axes[1].set_ylabel('Jumlah Populasi (Jiwa)')
axes[1].legend()

plt.tight_layout()
plt.savefig('kompartemen_seirv.png', dpi=300)
plt.show()